# Founders' Forge — Evaluation Notebook

This notebook reproduces the two core metrics reported in **Section 9.2 (Evaluation Results)** of the project documentation:

1. **JSON parse success** — can `extract_json()` reliably recover valid JSON from raw LLM output, even when it's wrapped in markdown fences or surrounded by commentary?
2. **Financial arithmetic accuracy** — does the Python-side `recompute_breakeven()` function correctly recalculate the breakeven figure from an agent's own stated inputs, fixing the circular-reasoning bug described in Section 9.3 (Failure Case 1)?

Run all cells top to bottom. Each cell prints a clear PASS / FAIL result.

## 1. `extract_json()` — recovering valid JSON from raw LLM output

In [1]:
import json
import re

def extract_json(raw_text: str):
    """
    Multi-strategy JSON extraction, matching the approach described in
    Section 7.2.2 of the project documentation:
      1. Try a direct parse (output is already clean JSON)
      2. Strip markdown code fences (```json ... ```) and retry
      3. Extract the first '{' to the last '}' in the text and retry
      4. Clean up trailing commas and retry
    Returns (parsed_dict, strategy_used) or (None, "failed") if nothing works.
    """
    # Strategy 1: direct parse
    try:
        return json.loads(raw_text), "direct parse"
    except json.JSONDecodeError:
        pass

    # Strategy 2: strip code fences
    fenced = re.sub(r"^```(json)?|```$", "", raw_text.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(fenced), "stripped code fences"
    except json.JSONDecodeError:
        pass

    # Strategy 3: extract first { to last }
    start, end = raw_text.find("{"), raw_text.rfind("}")
    if start != -1 and end != -1 and end > start:
        candidate = raw_text[start:end + 1]
        try:
            return json.loads(candidate), "brace extraction"
        except json.JSONDecodeError:
            pass
        # Strategy 4: trailing comma cleanup on the extracted candidate
        cleaned = re.sub(r",\s*([}\]])", r"\1", candidate)
        try:
            return json.loads(cleaned), "brace extraction + trailing comma cleanup"
        except json.JSONDecodeError:
            pass

    return None, "failed"


In [2]:
# Three realistic samples of what an LLM agent might actually return,
# based on the failure modes observed during development (Section 7.2.2 / 9.3).

sample_clean = '''{"startup_cost": 215000, "monthly_burn": 115000, "breakeven_months": 1.4}'''

sample_fenced = '''```json
{"startup_cost": 215000, "monthly_burn": 115000, "breakeven_months": 1.4}
```'''

sample_with_commentary = '''Here is the finance breakdown you requested:
{"startup_cost": 215000, "monthly_burn": 115000, "breakeven_months": 1.4,}
Let me know if you need anything else!'''

test_samples = {
    "Clean JSON": sample_clean,
    "Fenced JSON (```json ... ```)": sample_fenced,
    "JSON with surrounding commentary + trailing comma": sample_with_commentary,
}

print(f"{'Sample':45} {'Result':6} {'Strategy used'}")
print("-" * 90)

all_passed = True
for name, raw in test_samples.items():
    parsed, strategy = extract_json(raw)
    passed = parsed is not None and parsed.get("startup_cost") == 215000
    all_passed = all_passed and passed
    print(f"{name:45} {'PASS' if passed else 'FAIL':6} {strategy}")

print()
print("OVERALL:", "PASS — all 3 realistic output shapes parsed successfully" if all_passed else "FAIL — see above")


Sample                                        Result Strategy used
------------------------------------------------------------------------------------------
Clean JSON                                    PASS   direct parse
Fenced JSON (```json ... ```)                 PASS   stripped code fences
JSON with surrounding commentary + trailing comma PASS   brace extraction + trailing comma cleanup

OVERALL: PASS — all 3 realistic output shapes parsed successfully


## 2. `recompute_breakeven()` — verifying the financial arithmetic fix

This reproduces the exact scenario shown in **Figure B6** (Finance sheet, Adventure Travel Agency example) and **Failure Case 1** in Section 9.3, where the LLM's own arithmetic was internally inconsistent.

In [3]:
def recompute_breakeven(startup_cost, expected_pricing, assumed_customers_month_12, monthly_burn):
    """
    Deterministic breakeven recalculation, matching Section 7.2.5.
    Overrides whatever value the LLM produced, using the LLM's own stated inputs.
    """
    monthly_revenue = expected_pricing * assumed_customers_month_12
    surplus = monthly_revenue - monthly_burn

    if surplus <= 0:
        return "Not reached at this scale", "Burn exceeds revenue at the assumed scale."

    months = round(startup_cost / surplus, 1)
    note = (
        f"Recalculated in code: {startup_cost} startup cost / "
        f"({monthly_revenue} monthly revenue - {monthly_burn} monthly burn "
        f"= {surplus} surplus) = {months} months."
    )
    return months, note


In [4]:
# Real inputs from the Finance sheet (Figure B6): Adventure Travel Agency, India
startup_cost = 215000
expected_pricing = 4500
assumed_customers_month_12 = 60
monthly_burn = 115000

months, note = recompute_breakeven(startup_cost, expected_pricing, assumed_customers_month_12, monthly_burn)

print(note)
print()

expected_result = 1.4
passed = months == expected_result
print(f"Expected breakeven (from the running app, Figure B6): {expected_result} months")
print(f"Recomputed breakeven (this notebook):                 {months} months")
print()
print("PASS — matches the app's reported figure exactly" if passed else "FAIL — mismatch, investigate")


Recalculated in code: 215000 startup cost / (270000 monthly revenue - 115000 monthly burn = 155000 surplus) = 1.4 months.

Expected breakeven (from the running app, Figure B6): 1.4 months
Recomputed breakeven (this notebook):                 1.4 months

PASS — matches the app's reported figure exactly


In [5]:
# Regression check against the ORIGINAL bug (Failure Case 1, Section 9.3):
# an early Finance agent output claimed "16 months" using circular reasoning
# ("16 months at 200,000 monthly burn = 3,200,000 total cost"), instead of
# actually dividing startup cost by net monthly surplus.

buggy_llm_claim_months = 16

print(f"LLM\'s original (buggy) claim: {buggy_llm_claim_months} months")
print(f"Code-verified correct value:  {months} months")
print()
print("PASS — deterministic recalculation catches and overrides the LLM\'s incorrect arithmetic"
      if months != buggy_llm_claim_months else "FAIL")


LLM's original (buggy) claim: 16 months
Code-verified correct value:  1.4 months

PASS — deterministic recalculation catches and overrides the LLM's incorrect arithmetic


## 3. Summary — reproducing the Section 9.2 results table

In [6]:
import pandas as pd

results = pd.DataFrame([
    {
        "Metric": "JSON parse success",
        "Baseline / Before": "Raw output often unusable as-is (fences/commentary)",
        "This notebook": "3/3 realistic output shapes parsed successfully",
        "Achieved?": "Yes",
    },
    {
        "Metric": "Financial arithmetic accuracy",
        "Baseline / Before": f"LLM claimed {buggy_llm_claim_months} months (circular reasoning)",
        "This notebook": f"{months} months (code-verified, matches live app)",
        "Achieved?": "Yes",
    },
])

results


,Metric,Baseline / Before,This notebook,Achieved?
0,JSON parse success,Raw output often unusable as-is (fences/commen...,3/3 realistic output shapes parsed successfully,Yes
1,Financial arithmetic accuracy,LLM claimed 16 months (circular reasoning),"1.4 months (code-verified, matches live app)",Yes


## Scope note

This notebook verifies the two metrics that are directly reproducible from code in isolation: JSON-parsing robustness and financial-arithmetic correctness. It does **not** re-run the full CrewAI pipeline (which requires live OpenAI and Serper API calls and 1–2 minutes per run), and it does not attempt to re-verify source-citation coverage or geography accuracy, since those were assessed by manual review of full agent outputs rather than a single checkable calculation — see **Section 9.2** and **Section 9.3** of the project documentation for those results and their honest limitations.